## Part 0: Download Dataset

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # loads ROBOFLOW_API_KEY from .env file

from roboflow import Roboflow
import os

# authenticate with Roboflow using API key from .env
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])

# access the playing cards workspace and project
project = rf.workspace("0lauk0").project("playing-cards-muou8")

# select dataset version 10
version = project.version(10)

# download dataset in YOLOv8 format
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...


# Real-Time Playing Card Detection using YOLOv8

This notebook covers training a custom YOLOv8 model on a playing card dataset and running real-time detection on a webcam feed.

## Part 1: Training

In [14]:
from ultralytics import YOLO

In [ ]:
def main():
    model = YOLO("yolov8n.pt")  # nano model - fastest, good for real-time
    
    # train the model on the custom playing card dataset
    model.train(
        data="Playing-cards-10/data.yaml",  # path to dataset config
        epochs=3,      # number of training epochs (reduced for quick demo run)
        imgsz=640,     # input image size
        batch=16,      # batch size
        name="playing_card_detector"  # run name, used for saving results
    )

if __name__ == "__main__":
    main()

## Part 2: Real-Time Webcam Detection

In [17]:
import cv2
from ultralytics import YOLO

In [ ]:
# load the trained model weights
model = YOLO("best.pt")

In [ ]:
# open webcam (index 0) using Media Foundation backend
cap = cv2.VideoCapture(0, cv2.CAP_MSMF)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)   # set capture width
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)  # set capture height

False

In [ ]:
while True:
    ret, frame = cap.read()  # read a frame from webcam
    if not ret:
        break  # stop if frame wasn't captured successfully

    # run detection on the current frame
    results = model(frame, verbose=False, conf=0.5)
    annotated = results[0].plot()  # draw bounding boxes and labels on frame

    # calculate and display FPS
    fps = 1000 / results[0].speed['inference']
    cv2.putText(annotated, f"FPS: {fps:.1f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # show the annotated frame in a window
    cv2.imshow("Playing Card Detection - YOLOv8", annotated)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break  # press 'q' to quit

In [ ]:
# release the webcam and close all OpenCV windows
cap.release()
cv2.destroyAllWindows()